## Computing the frequency distribution

In [ ]:
import spacy
from collections import Counter,defaultdict
import csv
import os
import stanza
import spacy
import re
import random
import pandas as pd

nlp = spacy.load("en_core_web_sm", disable=["ner"])

# POS and lemma pipeline with your custom tagger model
pos_tagger_model = './Desktop/CHILDES-Parser/saved_models/pos/en_childes_charlm_tagger.pt'

# Dependency parse pipeline with your custom parser model
parser_model = './Desktop/CHILDES-Parser/saved_models/depparse/en_childes_charlm_parser.pt'
nlp_childes = stanza.Pipeline(
    lang='en',
    processors='tokenize,pos,lemma,depparse',
    use_gpu=True,  # Set to True if GPU is available
    pos_model_path=pos_tagger_model,
    depparse_model_path=parser_model
)

2025-11-22 11:38:09 INFO: Checking for updates to resources.json in case models have been updated.  Note: this behavior can be turned off with download_method=None or download_method=DownloadMethod.REUSE_RESOURCES
2025-11-22 11:38:09 INFO: Downloaded file to /Users/frapadovani/Desktop/extensive_reg/stanza/stanza/resources.json
2025-11-22 11:38:09 WARNING: Language en package default expects mwt, which has been added
2025-11-22 11:38:10 INFO: Loading these models for language: en (English):
| Processor | Package                 |
---------------------------------------
| tokenize  | combined                |
| mwt       | combined                |
| pos       | /Users/fra..._tagger.pt |
| lemma     | combined_nocharlm       |
| depparse  | /Users/fra..._parser.pt |

2025-11-22 11:38:10 WARNING: GPU requested, but is not available!
2025-11-22 11:38:10 INFO: Using device: cpu
2025-11-22 11:38:10 INFO: Loading: tokenize
2025-11-22 11:38:10 INFO: Loading: mwt
2025-11-22 11:38:10 INFO: Loadi

In [3]:
def compute_word_pos_tag_frequencies(file_path):
    """
    Compute frequency of (word, POS, tag) triples in a text file.
    Returns a Counter.
    """
    freq_counter = Counter()

    with open(file_path, "r", encoding="utf-8") as f:
        for line in f:
            line = line.strip()
            if not line:
                continue
            doc = nlp(line)
            for token in doc:
                if token.is_alpha:  # skip punctuation
                    triple = (token.text.lower(), token.pos_, token.tag_)
                    freq_counter[triple] += 1

    return freq_counter

def compute_word_pos_tag_frequencies_stanza(file_path):
    """
    Compute frequency of (word, POS, tag) triples in a text file.
    Returns a Counter.
    """
    freq_counter = Counter()

    with open(file_path, "r", encoding="utf-8") as f:
        for line in f:
            line = line.strip()
            if not line:
                continue
            doc = nlp_childes(line)
            for sentence in doc.sentences:
                for word in sentence.words:  # "words" has upos and xpos
                    if word.text.isalpha():  # skip punctuation/numbers
                        triple = (word.text.lower(), word.upos, word.xpos)
                        freq_counter[triple] += 1

    return freq_counter


def save_frequencies_to_csv(freq_counter, output_csv_path):
    """
    Save frequency counter to a CSV file.
    """
    with open(output_csv_path, "w", newline="", encoding="utf-8") as csvfile:
        writer = csv.writer(csvfile)
        writer.writerow(["word", "coarse_pos", "fine_tag", "frequency"])
        for (word, pos, tag), freq in freq_counter.items():
            writer.writerow([word, pos, tag, freq])

## BIN generation for CHILDES

In [ ]:
dataset_path = "./corpora/CHILDES_rand_new/original/childes_rand.train.txt"
output_csv = "./frequency_files/word_pos_tag_childes_stanza_rand_new.csv"

print("Processing dataset...")
freq_counter = compute_word_pos_tag_frequencies_stanza(dataset_path)
print(f"Found {len(freq_counter)} unique (word, POS, tag) triples.")

print(f"Saving frequencies to {output_csv}...")
save_frequencies_to_csv(freq_counter, output_csv)
print("Done!")

## BIN generation for WIKIPEDIA

In [ ]:
dataset_path = "./corpora/WIKI_rand/original/wiki_rand.train.txt"
output_csv = "./frequency_files/word_pos_tag_wiki_rand.csv"

print("Processing dataset...")
freq_counter = compute_word_pos_tag_frequencies(dataset_path)
print(f"Found {len(freq_counter)} unique (word, POS, tag) triples.")

print(f"Saving frequencies to {output_csv}...")
save_frequencies_to_csv(freq_counter, output_csv)
print("Done!")

## BIN generation for WRITTEN ADULT (BNC)

In [ ]:
dataset_path = "./corpora/BNC_rand/original/bnc_rand.train.txt"
output_csv = "./frequency_files/word_pos_tag_bnc_rand.csv"

print("Processing dataset...")
freq_counter = compute_word_pos_tag_frequencies(dataset_path)
print(f"Found {len(freq_counter)} unique (word, POS, tag) triples.")

print(f"Saving frequencies to {output_csv}...")
save_frequencies_to_csv(freq_counter, output_csv)
print("Done!")

## BIN generation for CANDOR

In [ ]:
dataset_path = "./Desktop/syntactic-bootstrapping/corpora/CANDOR_rand/original/candor.train.txt"
output_csv = "./Desktop/syntactic-bootstrapping/frequency_files/word_pos_tag_candor_new.csv"

print("Processing dataset...")
freq_counter = compute_word_pos_tag_frequencies(dataset_path)
print(f"Found {len(freq_counter)} unique (word, POS, tag) triples.")

print(f"Saving frequencies to {output_csv}...")
save_frequencies_to_csv(freq_counter, output_csv)
print("Done!")

Processing dataset...
Found 79028 unique (word, POS, tag) triples.
Saving frequencies to /Users/frapadovani/Desktop/syntactic-bootstrapping/frequency_files/word_pos_tag_candor_new.csv...
Done!


## Bin frequency creation

### Intervene by removing distributional co-occurrence associated with VERBS

Nouns, adjectives, and adverbs that co-occur with the verb in the sentence are replaced
with other words with the same part-of-speech (PoS).

For sentences with multiple verbs, the main verb remains unchanged, while the embedded verbs are sampled in the same
way as other PoS categories.

In [11]:
import os
import pandas as pd
import numpy as np

def assign_bins_and_save(df, output_dir="binned_data", num_bins=10):
    """
    Assigns words to logarithmic bins per (coarse_pos, fine_tag),
    and saves CSVs in a nested folder structure.
    """
    os.makedirs(output_dir, exist_ok=True)

    # Group by POS and fine-grained tag
    for (pos, tag), group in df.groupby(["coarse_pos", "fine_tag"]):
        freqs = group["frequency"].values
        if len(freqs) == 0:
            continue

        # Compute bin edges in log space
        log_min, log_max = np.log10(min(freqs)), np.log10(max(freqs))
        bin_edges = np.logspace(log_min, log_max, num_bins + 1)

        # Create folder structure: output_dir/POS/TAG/
        tag_folder = os.path.join(output_dir, pos, tag)
        os.makedirs(tag_folder, exist_ok=True)

        # Assign bins
        for bin_idx in range(num_bins):
            bin_rows = []
            for _, row in group.iterrows():
                freq = row["frequency"]
                b_idx = np.clip(np.digitize(freq, bin_edges, right=True) - 1, 0, num_bins - 1)

                if b_idx == bin_idx:
                    bin_rows.append(row.to_dict())

            # Save bin file if it has data
            if bin_rows:
                bin_df = pd.DataFrame(bin_rows)
                bin_file = os.path.join(tag_folder, f"bin_{bin_idx}.csv")
                bin_df.to_csv(bin_file, index=False)

## CHILDES binned data

In [ ]:
input_csv = "./frequency_files/word_pos_tag_childes_stanza_rand_new.csv"       
output_dir = "./binned_data/CHILDES_stanza_rand_new"

df = pd.read_csv(input_csv)
print(f"Loaded {len(df)} rows from {input_csv}")

assign_bins_and_save(df, output_dir=output_dir, num_bins=10)

print(f"Saved binned data into folder structure at {output_dir}")

## WIKIPEDIA binned data

In [ ]:
input_csv = "./frequency_files/word_pos_tag_wiki_rand.csv"       
output_dir = "./binned_data/WIKI_rand/"

df = pd.read_csv(input_csv)
print(f"Loaded {len(df)} rows from {input_csv}")

assign_bins_and_save(df, output_dir=output_dir, num_bins=10)

print(f"Saved binned data into folder structure at {output_dir}")

## ADULT WRITTEN-BNC binned data

In [ ]:
input_csv = "./frequency_files/word_pos_tag_bnc_rand.csv"       
output_dir = "./binned_data/BNC_rand/"

df = pd.read_csv(input_csv)
print(f"Loaded {len(df)} rows from {input_csv}")

assign_bins_and_save(df, output_dir=output_dir, num_bins=10)

print(f"Saved binned data into folder structure at {output_dir}")

## ADULT SPOKEN-CANDOR binned data

In [13]:
input_csv = "./frequency_files/new/word_pos_tag_candor_new.csv"    
output_dir = "./binned_data/BNC_SPOKEN_CANDOR_rand/"

df = pd.read_csv(input_csv)
print(f"Loaded {len(df)} rows from {input_csv}")

assign_bins_and_save(df, output_dir=output_dir, num_bins=10)

print(f"Saved binned data into folder structure at {output_dir}")

Loaded 79028 rows from ./frequency_files/new/word_pos_tag_candor_new.csv
Saved binned data into folder structure at ./binned_data/BNC_SPOKEN_CANDOR_rand/


## ACTUAL SUBSTITUTION 

In [14]:
def load_binned_words(binned_dir):
    """
    Load all words from the binned CSVs into a nested dictionary:
    binned_words[coarse_pos][fine_tag][bin_idx] = list of words
    """
    binned_words = defaultdict(lambda: defaultdict(lambda: defaultdict(list)))

    for pos in os.listdir(binned_dir):
        pos_path = os.path.join(binned_dir, pos)
        if not os.path.isdir(pos_path):
            continue
        for tag in os.listdir(pos_path):
            tag_path = os.path.join(pos_path, tag)
            if not os.path.isdir(tag_path):
                continue
            for bin_file in os.listdir(tag_path):
                if bin_file.endswith(".csv"):
                    bin_idx = int(bin_file.split("_")[1].split(".")[0])
                    df = pd.read_csv(os.path.join(tag_path, bin_file))
                    words = df["word"].tolist()
                    binned_words[pos][tag][bin_idx].extend(words)

    return binned_words

In [7]:
def get_main_and_embedded_verbs(doc):
    """
    Returns main verbs and embedded verbs using dependency parsing:
    - Main verb: ROOT of the sentence
    - Embedded verbs: other verbs dependent on ROOT
    """
    main_verbs = []
    embedded_verbs = []

    for token in doc:
        if token.pos_ == "VERB":
            if token.dep_ == "ROOT":
                main_verbs.append(token)
            elif token.head.pos_ == "VERB":  # dependent on another verb
                embedded_verbs.append(token)
    return main_verbs, embedded_verbs

In [8]:
import random
random.seed(42)

def replace_word(token, binned_words):
    pos, tag = token.pos_, token.tag_
    token_lower = token.text.lower()
    for bin_idx, words_in_bin in binned_words[pos][tag].items():
        if token_lower in words_in_bin:
            candidates = [w for w in words_in_bin if str(w).lower() != token_lower]
            return str(random.choice(candidates)).upper() if candidates else '_'+token.text+'_'
    return token.text.upper()

In [9]:
def augment_dataset(input_file, output_file, binned_words, batch_size=400):
    with open(input_file, "r", encoding="utf-8") as f_in:
        lines = [line.strip() for line in f_in if line.strip()]
        lines = lines  # Limit to first 10,000 lines for efficiency

    with open(output_file, "w", encoding="utf-8") as f_out:
        for doc in nlp.pipe(lines, batch_size=batch_size):
            main_verbs, embedded_verbs = get_main_and_embedded_verbs(doc)

            # Only augment if there is at least one main verb
            if main_verbs:
                new_tokens = []
                for token in doc:
                    if token.pos_ in ["NOUN", "ADJ", "ADV", "VERB"]:
                        # Keep main verbs unchanged
                        if token.pos_ == "VERB" and token in main_verbs:
                            new_tokens.append(token.text)
                        else:
                            new_tokens.append(replace_word(token, binned_words))
                    else:
                        new_tokens.append(token.text)
                f_out.write(" ".join(new_tokens) + "\n")
            else:
                # No main verb: write the original sentence
                f_out.write(doc.text + "\n")

In [10]:
def augment_dataset_stanza(input_file, output_file, binned_words, nlp_stanza, batch_size=400):
    """
    Augment a dataset using a Stanza pipeline.

    Args:
        input_file: path to input text file (one sentence per line)
        output_file: path to write augmented sentences
        binned_words: dictionary of replacement words
        nlp_stanza: a stanza.Pipeline object
        batch_size: not used here (Stanza doesn't support batch processing in the same way)
    """
    # Read input lines
    with open(input_file, "r", encoding="utf-8") as f_in:
        lines = [line.strip() for line in f_in if line.strip()]

    with open(output_file, "w", encoding="utf-8") as f_out:
        for line in lines:
            doc = nlp_stanza(line)  # Process one sentence at a time
            all_tokens = [word for sent in doc.sentences for word in sent.words]

            # Identify main and embedded verbs
            main_verbs = [w for w in all_tokens if w.upos == "VERB" and w.deprel == "root"]
            embedded_verbs = [w for w in all_tokens if w.upos == "VERB" and w.head != 0 and any(h.deprel == "root" for h in all_tokens if h.id == w.head)]

            # Only augment if at least one main verb
            if main_verbs:
                new_tokens = []
                for token in all_tokens:
                    if token.upos in ["NOUN", "ADJ", "ADV", "VERB"]:
                        # Keep main verbs unchanged
                        if token in main_verbs:
                            new_tokens.append(token.text)
                        else:
                            new_tokens.append(replace_word_stanza(token, binned_words))
                    else:
                        new_tokens.append(token.text)
                f_out.write(" ".join(new_tokens) + "\n")
            else:
                # No main verb: write original sentence
                f_out.write(line + "\n")


def replace_word_stanza(token, binned_words):
    """
    Replacement function for Stanza tokens (similar to your spaCy one)
    """
    pos, tag = token.upos, token.xpos
    token_lower = token.text.lower()
    for bin_idx, words_in_bin in binned_words[pos][tag].items():
        if token_lower in words_in_bin:
            candidates = [w for w in words_in_bin if str(w).lower() != token_lower]
            return str(random.choice(candidates)).upper() if candidates else '_' + token.text + '_'
    return token.text.upper()

In [ ]:
binned_dir = "./binned_data/CHILDES_stanza_rand"
binned_words = load_binned_words(binned_dir)

datasets = {
    "train": "./corpora/CHILDES_rand/original/childes_rand.train.txt",
    "dev": "./corpora/CHILDES_rand/original/childes_rand.dev.txt",
}

output_dir = "./corpora/CHILDES_rand/replace_word/"
os.makedirs(output_dir, exist_ok=True)

for split, input_file in datasets.items():
    output_file = os.path.join(output_dir, f"childes_rand.{split}.txt")
    print(f"Processing {split} split...")
    augment_dataset_stanza(input_file, output_file, binned_words, nlp_childes)
    print(f"Saved augmented {split} to {output_file}")

In [15]:
auxiliaries = [
    "am", "are", "is", "was", "were",
    "have", "has", "had",
    "do", "does", "did",
    "can", "could",
    "will", "would",
    "shall", "should",
    "may", "might",
    "must"
]

INSIDE_BRACKETS = re.compile(r"([\(\[\{])\s*(.*?)\s*([\)\]\}])")
INSIDE_QUOTES = re.compile(r"([\"'‘“”])\s*(.*?)\s*([\"'“’”])")
REMOVE_SPACE_BEFORE_PUNCT = re.compile(r"\s+([,.:;!?])")          # space before punctuation
punct_fix = re.compile(r"\s+([,.'?!])$")      # space before . ? !
ellipsis_fix = re.compile(r"\s+(\.\.\.)$")  # space before ...
underscores_fix = re.compile(r"_([A-Za-z0-9]+)_")  # remove surrounding underscores
aux_pattern = re.compile(r"\b(" + "|".join(auxiliaries) + r")\s+n['’]t\b", flags=re.IGNORECASE)
generic_clitic_pattern = re.compile(
    r"\b([A-Za-z]+)\s+('re|'m|'s|'ve|'d|'ll|n['’]t)\b", flags=re.IGNORECASE
)
# common split-to-joined contractions
split_to_joined = {
    r"\bwan\s+na\b": "wanna",
    r"\bgon\s+na\b": "gonna",
    r"\bgot\s+ta\b": "gotta",
    r"\blem\s+me\b": "lemme",
    r"\bgim\s+me\b": "gimme",
    r"\dun\s+no\s+know\b": "dunno",
    r"\bought\s+to\b": "oughta",
    r"\bcan\s+not\b": "cannot",
}
split_patterns = [(re.compile(k, flags=re.IGNORECASE), v) for k, v in split_to_joined.items()]
hyphen_fix_pattern = re.compile(r"\b([A-Za-z0-9]+)\s*-\s*([A-Za-z0-9]+)\b")



def lowercase_and_fix_punctuation(datasets: dict, output_dir: str):
    os.makedirs(output_dir, exist_ok=True)

    for split, input_file in datasets.items():
        base_name = os.path.splitext(os.path.basename(input_file))[0]
        output_file = os.path.join(output_dir, f"{base_name}.{split}.lower.txt")
        print(f"Processing {split} split for lowercase + punctuation + underscores fix...")

        with open(input_file, "r", encoding="utf-8") as f_in, open(output_file, "w", encoding="utf-8") as f_out:
            for line in f_in:
                line = line.strip().lower()
                # fix punctuation at end
                line = punct_fix.sub(r"\1", line)
                line = ellipsis_fix.sub(r"\1", line)
                # remove underscores around words
                line = underscores_fix.sub(r"\1", line)
                line = aux_pattern.sub(lambda m: m.group(1).lower() + "n't", line)
                line = generic_clitic_pattern.sub(lambda m: m.group(1) + m.group(2), line)
                for pattern, replacement in split_patterns:
                    line = pattern.sub(replacement, line)
                line = hyphen_fix_pattern.sub(r"\1-\2", line)
                if line.startswith("' "):
                    line = line.replace("' ", "'", 1)
                if line.startswith('" '):
                    line = line.replace('" ', '"', 1)
                if line.endswith('  . "'):
                    line = line.replace('  . "', '."', 1)
                if ' ...' in line: 
                    line = line.replace(' ...', '...', 1)
                if ' . ..' in line:
                    line = line.replace(' . ..', '...', 1)
                if ' . . .' in line:
                    line = line.replace(' . . .', '...', 1)
                line = REMOVE_SPACE_BEFORE_PUNCT.sub(r"\1", line)
                line = INSIDE_BRACKETS.sub(r"\1\2\3", line)
                line = INSIDE_QUOTES.sub(r"\1\2\3", line)
                line = line.replace("  ", " ")
                f_out.write(line + "\n")
                

        print(f"Saved fixed {split} to {output_file}")

## CHILDES verbs 

In [ ]:
datasets = {"train": "./corpora/CHILDES_rand/replace_word/childes_rand.train.txt",
    "dev": "./corpora/CHILDES_rand/replace_word/childes_rand.dev.txt"}
lowercase_and_fix_punctuation(datasets, "./corpora/CHILDES_rand/replace_word/")

Processing train split for lowercase + punctuation + underscores fix...
Saved fixed train to ./corpora/CHILDES_rand/replace_word/childes_rand.train.train.lower.txt
Processing dev split for lowercase + punctuation + underscores fix...
Saved fixed dev to ./corpora/CHILDES_rand/replace_word/childes_rand.dev.dev.lower.txt


## CHILDES nouns

In [ ]:
datasets = {"train": "./corpora/CHILDES/replace_word_stanza_noun/childes.train.txt",
    "dev": "./corpora/CHILDES/replace_word_stanza_noun/childes.dev.txt"}

lowercase_and_fix_punctuation(datasets, "./corpora/CHILDES/replace_word_stanza_noun/")

Processing train split for lowercase + punctuation + underscores fix...
Saved fixed train to ./corpora/CHILDES/replace_word_stanza_noun/childes.train.train.lower.txt
Processing dev split for lowercase + punctuation + underscores fix...
Saved fixed dev to ./corpora/CHILDES/replace_word_stanza_noun/childes.dev.dev.lower.txt


In [ ]:
binned_dir = "./binned_data/CHILDES_stanza_rand"
binned_words = load_binned_words(binned_dir)

datasets = {
    "train": "./corpora/CHILDES_rand/original/childes_rand.train.txt",
    "dev": "./corpora/CHILDES_rand/original/childes_rand.dev.txt",
}

output_dir = "./corpora/CHILDES_rand/replace_word/"
os.makedirs(output_dir, exist_ok=True)

for split, input_file in datasets.items():
    output_file = os.path.join(output_dir, f"childes_rand.{split}.txt")
    print(f"Processing {split} split...")
    augment_dataset_stanza(input_file, output_file, binned_words,nlp_childes)
    print(f"Saved augmented {split} to {output_file}")

Processing train split...


## WIKIPEDIA

In [17]:
binned_dir = "./binned_data/WIKI_rand"
binned_words = load_binned_words(binned_dir)

datasets = {
    "train": "./corpora/WIKI_rand/original/wiki_rand.train.txt",
    "dev": "./corpora/WIKI_rand/original/wiki_rand.dev.txt",
}

output_dir = "./corpora/WIKI_rand/replace_word/"
os.makedirs(output_dir, exist_ok=True)

for split, input_file in datasets.items():
    output_file = os.path.join(output_dir, f"wiki_rand.{split}.txt")
    print(f"Processing {split} split...")
    augment_dataset(input_file, output_file, binned_words, batch_size=10)
    print(f"Saved augmented {split} to {output_file}")

Processing train split...
Saved augmented train to ./corpora/WIKI_rand/replace_word/wiki_rand.train.txt
Processing dev split...
Saved augmented dev to ./corpora/WIKI_rand/replace_word/wiki_rand.dev.txt


In [12]:
## APPLY THE LOWERCASE AND FIX PUNCTUATION ALSO TO THE ORIGINAL DATASET, NOT ONLY TO THE REPLACE WORD

In [ ]:
datasets = {"train": "./corpora/WIKI/replace_word/wiki.train.txt",
    "dev": "./corpora/WIKI/replace_word/wiki.dev.txt",
    "test": "./corpora/WIKI/replace_word/wiki.test.txt"}

lowercase_and_fix_punctuation(datasets, "./corpora/WIKI/replace_word/")

Processing train split for lowercase + punctuation + underscores fix...
Saved fixed train to ./corpora/WIKI/replace_word/wiki.train.train.lower.txt
Processing dev split for lowercase + punctuation + underscores fix...
Saved fixed dev to ./corpora/WIKI/replace_word/wiki.dev.dev.lower.txt
Processing test split for lowercase + punctuation + underscores fix...
Saved fixed test to ./corpora/WIKI/replace_word/wiki.test.test.lower.txt


In [ ]:
datasets = {"train": "./corpora/WIKI/shuffle_order/np_final/wiki_np.train.txt",
    "dev": "./corpora/WIKI/shuffle_order/np_final/wiki_np.dev.txt"}

lowercase_and_fix_punctuation(datasets, "./corpora/WIKI/shuffle_order/np_final/")

Processing train split for lowercase + punctuation + underscores fix...
Saved fixed train to ./corpora/WIKI/shuffle_order/np_final/wiki_np.train.train.lower.txt
Processing dev split for lowercase + punctuation + underscores fix...
Saved fixed dev to ./corpora/WIKI/shuffle_order/np_final/wiki_np.dev.dev.lower.txt


## WRITTEN ADULT

In [ ]:
binned_dir = "./binned_data/WRITTEN_ADULT"
binned_words = load_binned_words(binned_dir)

datasets = {
    "train": "./corpora/BNC/original/bnc.train.txt",
    "dev": "./corpora/BNC/original/bnc.dev.txt",
    "test": "./corpora/BNC/original/bnc.test.txt"
}

output_dir = "./corpora/BNC/replace_word/"
os.makedirs(output_dir, exist_ok=True)

for split, input_file in datasets.items():
    output_file = os.path.join(output_dir, f"bnc.{split}.txt")
    print(f"Processing {split} split...")
    augment_dataset(input_file, output_file, binned_words, batch_size=10)
    print(f"Saved augmented {split} to {output_file}")

In [35]:
binned_dir = "./binned_data/BNC_rand"
binned_words = load_binned_words(binned_dir)

datasets = {
    "train": "./corpora/BNC_rand/original/bnc_rand.train.txt"
}

output_dir = "./corpora/BNC_rand/replace_word/"
os.makedirs(output_dir, exist_ok=True)

for split, input_file in datasets.items():
    output_file = os.path.join(output_dir, f"bnc_rand.{split}.txt")
    print(f"Processing {split} split...")
    augment_dataset(input_file, output_file, binned_words, batch_size=10)
    print(f"Saved augmented {split} to {output_file}")

Processing train split...
Saved augmented train to ./corpora/BNC_rand/replace_word/bnc_rand.train.txt


## SPOKEN ADULT

In [16]:
binned_dir = "./binned_data/BNC_SPOKEN_CANDOR_rand"
binned_words = load_binned_words(binned_dir)

datasets = {
    "train": "./corpora/CANDOR_rand/original/candor.train.txt",
    "dev": "./corpora/CANDOR_rand/original/candor.dev.txt",
    "test": "./corpora/CANDOR_rand/original/candor.test.txt"
}

output_dir = "./corpora/CANDOR_rand/replace_word/"
os.makedirs(output_dir, exist_ok=True)

for split, input_file in datasets.items():
    output_file = os.path.join(output_dir, f"candor.{split}.txt")
    print(f"Processing {split} split...")
    augment_dataset(input_file, output_file, binned_words, batch_size=10)
    print(f"Saved augmented {split} to {output_file}")

Processing train split...
Saved augmented train to ./corpora/CANDOR_rand/replace_word/candor.train.txt
Processing dev split...
Saved augmented dev to ./corpora/CANDOR_rand/replace_word/candor.dev.txt
Processing test split...
Saved augmented test to ./corpora/CANDOR_rand/replace_word/candor.test.txt


In [17]:
datasets = {
    "train": "./corpora/CANDOR_rand/replace_word/candor.train.txt",
    "dev": "./corpora/CANDOR_rand/replace_word/candor.dev.txt",
    "test": "./corpora/CANDOR_rand/replace_word/candor.test.txt"
}
lowercase_and_fix_punctuation(datasets, "./corpora/CANDOR_rand/replace_word/")

Processing train split for lowercase + punctuation + underscores fix...
Saved fixed train to ./corpora/CANDOR_rand/replace_word/candor.train.train.lower.txt
Processing dev split for lowercase + punctuation + underscores fix...
Saved fixed dev to ./corpora/CANDOR_rand/replace_word/candor.dev.dev.lower.txt
Processing test split for lowercase + punctuation + underscores fix...
Saved fixed test to ./corpora/CANDOR_rand/replace_word/candor.test.test.lower.txt


In [ ]:
datasets = {
    "train": "./corpora/BNC_SPOKEN/replace_word/bnc_spoken.train.txt",
    "dev": "./corpora/BNC_SPOKEN/replace_word/bnc_spoken.dev.txt",
    "test": "./corpora/BNC_SPOKEN/replace_word/bnc_spoken.test.txt"
}
lowercase_and_fix_punctuation(datasets, "./corpora/BNC_SPOKEN/replace_word/")

Processing train split for lowercase + punctuation + underscores fix...
Saved fixed train to ./corpora/SPOKEN_ADULT/replace_word/bnc_spoken.train.train.lower.txt
Processing dev split for lowercase + punctuation + underscores fix...
Saved fixed dev to ./corpora/SPOKEN_ADULT/replace_word/bnc_spoken.dev.dev.lower.txt
Processing test split for lowercase + punctuation + underscores fix...
Saved fixed test to ./corpora/SPOKEN_ADULT/replace_word/bnc_spoken.test.test.lower.txt


In [ ]:
datasets = {
    "train": "./corpora/BNC_SPOKEN/replace_word_noun/bnc_spoken.train.txt",
    "dev": "./corpora/BNC_SPOKEN/replace_word_noun/bnc_spoken.dev.txt",
    "test": "./corpora/BNC_SPOKEN/replace_word_noun/bnc_spoken.test.txt"
}
lowercase_and_fix_punctuation(datasets, "./corpora/BNC_SPOKEN/replace_word_noun/")

## SPOKEN ADULT CANDOR

In [12]:
binned_dir = "./binned_data/BNC_SPOKEN_CANDOR"
binned_words = load_binned_words(binned_dir)

datasets = {
    #"train": "./corpora/BNC_SPOKEN_CANDOR/original/candor.train.txt",
    "dev": "./corpora/BNC_SPOKEN_CANDOR/original/candor.dev.txt",
}

output_dir = "./corpora/BNC_SPOKEN_CANDOR/replace_word/"
os.makedirs(output_dir, exist_ok=True)

for split, input_file in datasets.items():
    output_file = os.path.join(output_dir, f"candor.{split}.txt")
    print(f"Processing {split} split...")
    augment_dataset(input_file, output_file, binned_words, batch_size=10)
    print(f"Saved augmented {split} to {output_file}")

Processing dev split...
Saved augmented dev to ./corpora/BNC_SPOKEN_CANDOR/replace_word/candor.dev.txt


# Extra experiments on NOUN REPLACEMENT

In [16]:
def augment_dataset_noun_focus(input_file, output_file, binned_words, batch_size=400):
    """
    Replace nouns, adjectives, and adverbs that co-occur with a noun in the sentence.
    If more than one noun, keep one randomly unchanged.
    """
    with open(input_file, "r", encoding="utf-8") as f_in:
        lines = [line.strip() for line in f_in if line.strip()]

    with open(output_file, "w", encoding="utf-8") as f_out:
        for doc in nlp.pipe(lines, batch_size=batch_size):
            # Collect all nouns
            nouns = [token for token in doc if token.pos_ == "NOUN"]

            if not nouns:
                # No nouns: write original sentence
                f_out.write(doc.text + "\n")
                continue

            # Randomly choose one noun to remain unchanged if multiple nouns
            noun_to_keep = random.choice(nouns) if len(nouns) > 1 else nouns[0]

            new_tokens = []
            for token in doc:
                if token.pos_ in ["VERB", "ADJ", "ADV", "NOUN"]:
                    # Keep the chosen noun unchanged
                    if token == noun_to_keep:
                        new_tokens.append(token.text)
                    else:
                        new_tokens.append(replace_word(token, binned_words))
                else:
                    new_tokens.append(token.text)

            f_out.write(" ".join(new_tokens) + "\n")


In [13]:
def augment_dataset_noun_focus_stanza(input_file, output_file, binned_words):
    """
    Replace nouns, adjectives, adverbs, and verbs that co-occur with a noun in the sentence.
    If more than one noun, keep one randomly unchanged.
    Adapted for Stanza parser.
    """

    # Read input sentences
    with open(input_file, "r", encoding="utf-8") as f_in:
        lines = [line.strip() for line in f_in if line.strip()]

    with open(output_file, "w", encoding="utf-8") as f_out:
        for sentence in lines:
            doc = nlp_childes(sentence)  # one sentence at a time

            for sent in doc.sentences:
                # Collect nouns
                nouns = [token for token in sent.tokens if token.words[0].upos == "NOUN"]

                if not nouns:
                    f_out.write(" ".join([t.text for t in sent.tokens]) + "\n")
                    continue

                noun_to_keep = random.choice(nouns) if len(nouns) > 1 else nouns[0]

                new_tokens = []
                for token in sent.tokens:
                    word_obj = token.words[0]
                    if word_obj.upos in ["NOUN", "ADJ", "ADV", "VERB"]:
                        if token == noun_to_keep:
                            new_tokens.append(token.text)
                        else:
                            new_tokens.append(replace_word_stanza(word_obj, binned_words))
                    else:
                        new_tokens.append(token.text)

                f_out.write(" ".join(new_tokens) + "\n")
    

## CHILDES

In [ ]:
binned_dir = "./binned_data/CHILDES_spacy"
binned_words = load_binned_words(binned_dir)

datasets = {
    "train": "./corpora/CHILDES/original/childes.train.txt",
    "dev": "./corpora/CHILDES/original/childes.dev.txt",
}

output_dir = "./corpora/CHILDES/replace_word_spacy_noun/"
os.makedirs(output_dir, exist_ok=True)

for split, input_file in datasets.items():
    output_file = os.path.join(output_dir, f"childes.{split}.txt")
    print(f"Processing {split} split...")
    augment_dataset_noun_focus(input_file, output_file, binned_words)
    print(f"Saved augmented {split} to {output_file}")

In [ ]:
binned_dir = "./binned_data/CHILDES_stanza_rand"
binned_words = load_binned_words(binned_dir)

datasets = {
    "train": "./corpora/CHILDES_rand/original/childes_rand.train.txt",
    "dev": "./corpora/CHILDES_rand/original/childes_rand.dev.txt",
}

output_dir = "./corpora/CHILDES_rand/replace_word_noun/"
os.makedirs(output_dir, exist_ok=True)

for split, input_file in datasets.items():
    output_file = os.path.join(output_dir, f"childes_rand.{split}.txt")
    print(f"Processing {split} split...")
    augment_dataset_noun_focus_stanza(input_file, output_file, binned_words)
    print(f"Saved augmented {split} to {output_file}")

Processing train split...


## WIKI

In [ ]:
binned_dir = "./binned_data/WIKI_rand"
binned_words = load_binned_words(binned_dir)

datasets = {
    "train": "./corpora/WIKI_rand/original/wiki_rand.train.txt",
    "dev": "./corpora/WIKI_rand/original/wiki_rand.dev.txt",
}

output_dir = "./corpora/WIKI_rand/replace_word_noun/"
os.makedirs(output_dir, exist_ok=True)

for split, input_file in datasets.items():
    output_file = os.path.join(output_dir, f"wiki_rand.{split}.txt")
    print(f"Processing {split} split...")
    augment_dataset_noun_focus(input_file, output_file, binned_words, batch_size=10)
    print(f"Saved augmented {split} to {output_file}")

In [ ]:
datasets = {
    "train": "./corpora/WIKI_rand/replace_word_noun/wiki_rand.train.txt",
    "dev": "./corpora/WIKI_rand/replace_word_noun/wiki_rand.dev.txt",
}
lowercase_and_fix_punctuation(datasets, "./corpora/WIKI_rand/replace_word_noun/")

## BNC WRITTEN

In [ ]:
binned_dir = "./binned_data/BNC_rand"
binned_words = load_binned_words(binned_dir)

datasets = {
    "train": "./corpora/BNC_rand/original/bnc_rand.train.txt",
    "dev": "./corpora/BNC_rand/original/bnc_rand.dev.txt",
}

output_dir = "./corpora/BNC_rand/replace_word_noun/"
os.makedirs(output_dir, exist_ok=True)

for split, input_file in datasets.items():
    output_file = os.path.join(output_dir, f"bnc_rand.{split}.txt")
    print(f"Processing {split} split...")
    augment_dataset_noun_focus(input_file, output_file, binned_words, batch_size=10)
    print(f"Saved augmented {split} to {output_file}")

## BNC SPOKEN

In [ ]:
binned_dir = "./binned_data/BNC_SPOKEN"
binned_words = load_binned_words(binned_dir)

datasets = {
    "train": "./corpora/BNC_SPOKEN/original/bnc_spoken.train.txt",
    "dev": "./corpora/BNC_SPOKEN/original/bnc_spoken.dev.txt",
    "test": "./corpora/BNC_SPOKEN/original/bnc_spoken.test.txt"
}

output_dir = "./corpora/BNC_SPOKEN/replace_word_noun/"
os.makedirs(output_dir, exist_ok=True)

for split, input_file in datasets.items():
    output_file = os.path.join(output_dir, f"bnc_spoken.{split}.txt")
    print(f"Processing {split} split...")
    augment_dataset_noun_focus(input_file, output_file, binned_words, batch_size=10)
    print(f"Saved augmented {split} to {output_file}")

Processing train split...
Saved augmented train to /home/p318482/syntactic-bootstrapping/corpora/BNC_SPOKEN/replace_word_noun/bnc_spoken.train.txt
Processing dev split...
Saved augmented dev to /home/p318482/syntactic-bootstrapping/corpora/BNC_SPOKEN/replace_word_noun/bnc_spoken.dev.txt
Processing test split...
Saved augmented test to /home/p318482/syntactic-bootstrapping/corpora/BNC_SPOKEN/replace_word_noun/bnc_spoken.test.txt


## Check proportion of words replaced for datasets

In [ ]:
import glob

def proportion_replaced_tokens(file_path):
    """
    Calculate the average proportion of replaced tokens (CAPITALIZED) in a file.
    Each line is treated as a sentence.
    """
    total_proportions = []
    with open(file_path, "r", encoding="utf-8") as f:
        for line in f:
            line = line.strip()
            if not line:
                continue
            tokens = line.split()
            num_tokens = len(tokens)
            if num_tokens == 0:
                continue
            # Count tokens that are fully uppercase (replaced)
            replaced_tokens = sum(1 for t in tokens if t.isupper())
            proportion = replaced_tokens / num_tokens
            total_proportions.append(proportion)
    # Average proportion across all sentences
    if total_proportions:
        return sum(total_proportions) / len(total_proportions)
    else:
        return 0.0

# List of files
files = [
    "./corpora/BNC/replace_word/bnc.train.upper.txt",
    "./corpora/BNC_rand/replace_word/bnc_rand.upper.train.txt",
    "./corpora/BNC_SPOKEN_CANDOR/replace_word/candor.train.upper.txt",
    "./corpora/CHILDES/replace_word/childes.train.upper.txt",
    "./corpora/CHILDES_rand/replace_word/childes_rand.upper.train.txt",
    "./corpora/WIKI/replace_word/wiki.train.upper.txt",
    "./corpora/WIKI_rand/replace_word/wiki_rand.upper.train.txt"
]

# Calculate and print average proportions
for file in files:
    avg_prop = proportion_replaced_tokens(file)
    print(f"{file}: Average replaced token proportion = {avg_prop:.4f}")


/home/p318482/syntactic-bootstrapping/corpora/BNC/replace_word/bnc.train.upper.txt: Average replaced token proportion = 0.2392
/home/p318482/syntactic-bootstrapping/corpora/BNC_rand/replace_word/bnc_rand.upper.train.txt: Average replaced token proportion = 0.2435
/home/p318482/syntactic-bootstrapping/corpora/BNC_SPOKEN_CANDOR/replace_word/candor.train.upper.txt: Average replaced token proportion = 0.1637
/home/p318482/syntactic-bootstrapping/corpora/CHILDES/replace_word/childes.train.upper.txt: Average replaced token proportion = 0.1032
/home/p318482/syntactic-bootstrapping/corpora/CHILDES_rand/replace_word/childes_rand.upper.train.txt: Average replaced token proportion = 0.1016
/home/p318482/syntactic-bootstrapping/corpora/WIKI/replace_word/wiki.train.upper.txt: Average replaced token proportion = 0.2631
/home/p318482/syntactic-bootstrapping/corpora/WIKI_rand/replace_word/wiki_rand.upper.train.txt: Average replaced token proportion = 0.2631
